# 02 Data Preparation


## 1. Data Inputs


In [ ]:

from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

NEWS_PATH = RAW_DIR / "polygon_news_sample.json"
TARGET_TICKERS = ["AAPL", "MSFT", "TSLA", "AMZN", "NVDA"]
BENCHMARK_TICKER = "SPY"
START_DATE = "2023-01-01"
END_DATE = "2024-01-01"

## 2. Load and Flatten News


In [ ]:
with NEWS_PATH.open("r", encoding="utf-8") as f:
    news_raw = json.load(f)

rows = []
for article in news_raw:
    for insight in article.get("insights", []):
        title = article.get("title") or ""
        description = article.get("description") or ""
        published_utc = article.get("published_utc")
        rows.append(
            {
                "article_id": article.get("id"),
                "published_utc": published_utc,
                "date": pd.to_datetime(published_utc, utc=True).date().isoformat(),
                "ticker": insight.get("ticker"),
                "existing_sentiment": insight.get("sentiment"),
                "title": title,
                "description": description,
                "text": f"{title}. {description}".strip(),
                "publisher": (article.get("publisher") or {}).get("name"),
                "article_url": article.get("article_url"),
                "sentiment_reasoning": insight.get("sentiment_reasoning"),
            }
        )

news_flat = pd.DataFrame(rows)
news_target = news_flat[news_flat["ticker"].isin(TARGET_TICKERS)].copy()

news_flat.to_csv(PROCESSED_DIR / "news_flattened.csv", index=False)
news_target.to_csv(PROCESSED_DIR / "news_target_tickers.csv", index=False)

news_flat.shape, news_target.shape

In [ ]:
coverage = (
    news_target.groupby("ticker")
    .agg(
        rows=("article_id", "count"),
        unique_news_days=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

coverage.to_csv(TABLES_DIR / "target_ticker_news_coverage.csv", index=False)
coverage

## 3. Download and Cache Price Data


In [ ]:
import yfinance as yf

price_tickers = TARGET_TICKERS + [BENCHMARK_TICKER]
price_frames = []

for ticker in price_tickers:
    px = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=False)
    px = px.reset_index()
    px.columns = [str(c).lower().replace(" ", "_") for c in px.columns]
    px["ticker"] = ticker
    px = px.rename(columns={"date": "date", "adj_close": "adj_close"})
    price_frames.append(px[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]])

prices = pd.concat(price_frames, ignore_index=True)
prices["date"] = pd.to_datetime(prices["date"]).dt.date.astype(str)
prices.to_csv(PROCESSED_DIR / "prices_2023.csv", index=False)

prices.groupby("ticker").size().reset_index(name="rows")

## 4. Price Features and Target


In [ ]:
def add_price_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["ticker", "date"]).copy()
    g = df.groupby("ticker", group_keys=False)

    df["return_1d"] = g["close"].pct_change(1)
    df["return_3d"] = g["close"].pct_change(3)
    df["return_5d"] = g["close"].pct_change(5)
    df["volatility_5d"] = g["return_1d"].rolling(5).std().reset_index(level=0, drop=True)
    df["volume_change_1d"] = g["volume"].pct_change(1)
    df["moving_avg_5d"] = g["close"].rolling(5).mean().reset_index(level=0, drop=True)
    df["moving_avg_20d"] = g["close"].rolling(20).mean().reset_index(level=0, drop=True)
    df["ma_5_20_gap"] = df["moving_avg_5d"] / df["moving_avg_20d"] - 1
    df["next_close"] = g["close"].shift(-1)
    df["target_next_day_up"] = (df["next_close"] > df["close"]).astype("Int64")
    df.loc[df["next_close"].isna(), "target_next_day_up"] = pd.NA
    return df.drop(columns=["next_close"])

price_features = add_price_features(prices)
price_features.to_csv(PROCESSED_DIR / "price_features_2023.csv", index=False)
price_features.head()

## 5. Temporary Daily Sentiment Proxy


In [ ]:
sentiment_value = {"positive": 1, "neutral": 0, "negative": -1, "mixed": 0}
tmp = news_target.copy()
tmp["sentiment_value"] = tmp["existing_sentiment"].map(sentiment_value).fillna(0)

daily_sentiment = (
    tmp.groupby(["ticker", "date"])
    .agg(
        news_count=("article_id", "count"),
        existing_positive_share=("existing_sentiment", lambda s: (s == "positive").mean()),
        existing_negative_share=("existing_sentiment", lambda s: (s == "negative").mean()),
        existing_neutral_share=("existing_sentiment", lambda s: (s == "neutral").mean()),
        existing_sentiment_score_mean=("sentiment_value", "mean"),
    )
    .reset_index()
)

daily_sentiment.to_csv(PROCESSED_DIR / "daily_existing_sentiment_features.csv", index=False)
daily_sentiment.head()

## 6. Merge Final Modelling Dataset


In [ ]:
stock_features = price_features[price_features["ticker"].isin(TARGET_TICKERS)].copy()
spy_features = price_features[price_features["ticker"] == BENCHMARK_TICKER][
    ["date", "return_1d", "return_5d", "volatility_5d"]
].rename(
    columns={
        "return_1d": "spy_return_1d",
        "return_5d": "spy_return_5d",
        "volatility_5d": "spy_volatility_5d",
    }
)

model_df = stock_features.merge(daily_sentiment, on=["ticker", "date"], how="left")
model_df = model_df.merge(spy_features, on="date", how="left")
model_df = model_df.rename(columns={"date": "trading_date"})

sentiment_cols = [
    "news_count",
    "existing_positive_share",
    "existing_negative_share",
    "existing_neutral_share",
    "existing_sentiment_score_mean",
]
model_df["has_news"] = model_df["news_count"].notna().astype(int)
model_df[sentiment_cols] = model_df[sentiment_cols].fillna(0)

model_df.to_csv(PROCESSED_DIR / "model_dataset_existing_sentiment_proxy.csv", index=False)

complete_model_df = model_df.dropna(
    subset=["return_5d", "moving_avg_20d", "spy_return_5d", "target_next_day_up"]
).copy()
complete_model_df.to_csv(PROCESSED_DIR / "model_dataset_existing_sentiment_proxy_complete.csv", index=False)

summary = pd.DataFrame(
    [
        {
            "dataset": "model_dataset_existing_sentiment_proxy",
            "rows": len(model_df),
            "complete_rows": len(complete_model_df),
            "rows_with_news": int(model_df["has_news"].sum()),
            "tickers": ", ".join(TARGET_TICKERS),
        }
    ]
)
summary.to_csv(TABLES_DIR / "model_dataset_summary.csv", index=False)
summary